In [0]:
# Load bronze tables
df_transactions = spark.table("databricks_banking_dev_ws2.bronze.bank_transaction_1")
df_branches = spark.table("databricks_banking_dev_ws2.bronze.branch_reference")

print(f"Total transactions: {df_transactions.count()}")
print(f"Total branches: {df_branches.count()}")

df_transactions.printSchema()

In [0]:
from pyspark.sql.functions import col, expr

df_silver_transactions = (
    df_transactions
    .withColumn("CustomerDOB", expr("try_to_date(CustomerDOB, 'dd-MM-yyyy')"))
    .filter(col("CustomerDOB").isNotNull())
    .filter(col("CustomerDOB") > "1900-01-01")   # remove invalid/very old DOBs like 1800
    .filter(col("CustAccountBalance").isNotNull())
    .filter(col("`TransactionAmount (INR)`").isNotNull())
    .withColumnRenamed("TransactionAmount (INR)", "TransactionAmount")
    .dropDuplicates(["TransactionID"])
)

print(f"Rows after cleaning: {df_silver_transactions.count()}")
df_silver_transactions.display()

In [0]:
from pyspark.sql.functions import col, to_date

df_silver_branches = (
    df_branches
    .withColumn("Opening_Date", to_date(col("Opening_Date"), "yyyy-MM-dd"))
    .dropDuplicates(["Branch_ID"])
)

df_silver_branches.display()

In [0]:
df_silver_transactions.write.format("delta").mode("overwrite").saveAsTable(
    "databricks_banking_dev_ws2.silver.customer_transactions_clean"
)

df_silver_branches.write.format("delta").mode("overwrite").saveAsTable(
    "databricks_banking_dev_ws2.silver.branch_reference_clean"
)

print("Silver tables created successfully!")